# Pruebas Objetivas

Estudiante: Luz Angela Rojas Prieto

Actividad 1: Preprocesamiento y Tokenización

In [2]:
# Paso 1: Importar librerías necesarias
import pandas as pd
import numpy as np
import re
import string
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as pl

In [3]:
# Descargar recursos de NLTK
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
from google.colab import drive
drive.mount('/content/drive')
# Ruta al archivo
ruta_archivo = '/content/drive/My Drive/Colab Notebooks/Noticias.xlsx'

# Leer el archivo Excel
data = pd.read_excel(ruta_archivo)

# Mostrar las primeras filas
data.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Columna1,Enlaces,Título,info,contenido,Etiqueta
0,0,https://www.eltiempo.com/agresion-contra-un-op...,Operador de grúa quedó inconsciente tras agres...,El conductor de una moto le lanzó el casco y p...,Las autoridades están buscando al conductor de...,colombia
1,1,https://www.eltiempo.com/archivo/documento/CMS...,"Usaquén, primera en infracciones por mal parqueo",La localidad ocupa el primer lugar en comparen...,"""Los andenes son para los peatones"", reclama e...",archivo
2,2,https://www.eltiempo.com/archivo/documento/CMS...,'Me atracaron y vi un arma que me heló la sang...,Un ciudadano relata cómo cuatro hombres lo rob...,A las 7 de la noche me había quedado de encont...,archivo
3,3,https://www.eltiempo.com/archivo/documento/CMS...,"Escoltas mal estacionados, dolor de cabeza de ...",Las zonas de restaurantes se convierten en par...,Atravesados. Eso es lo que se les pasa por la ...,archivo
4,4,https://www.eltiempo.com/archivo/documento/CMS...,Radicado primer proyecto que autorizaría union...,"El representante de 'la U', Miguel Gómez, dijo...",“Estamos proponiendo la figura de un contrato ...,archivo


In [5]:
# Eliminar filas con nulos en columnas específicas
data= data.dropna(subset=['Título', 'contenido','info', 'Etiqueta'])
print(f"Dataset después de eliminar nulos: {data.shape}")

Dataset después de eliminar nulos: (11483, 6)


Paso 1: Preprocesamiento del Texto

In [6]:
import pandas as pd
import re
import string

# Función para eliminar tildes y preservar la letra 'ñ'
def remove_accents(text):
    """
    Elimina las tildes del texto pero preserva la letra 'ñ'.
    """
    tildes = str.maketrans('áéíóúüÁÉÍÓÚÜ', 'aeiouuAEIOUU')
    text = text.translate(tildes)
    return text

# Función completa de preprocesamiento
def preprocess_text(text):
    """
    Preprocesamiento del texto:
    - Convierte el texto a minúsculas.
    - Elimina las tildes.
    - Elimina guiones y puntuación.
    - Elimina los números.
    - Elimina los espacios en blanco adicionales.
    """
    if not isinstance(text, str):
        return text
    # Convertir a minúsculas
    text = text.lower()
    # Quitar tildes
    text = remove_accents(text)
    # Eliminar guiones (normales y largos) y puntuación
    text = re.sub(r'[-–—]', '', text)  # Elimina guiones de cualquier tipo
    text = text.translate(str.maketrans('', '', string.punctuation + '‘’“”'))
    # Eliminar números
    text = re.sub(r'\d+', '', text)
    # Eliminar espacios en blanco adicionales
    text = ' '.join(text.split())
    return text

# Aplicar preprocesamiento
data['contenido_preprocesado'] = data['contenido'].apply(preprocess_text)

# Mostrar resultados
data[['contenido', 'contenido_preprocesado']].sample(25)


,contenido,contenido_preprocesado
2755,"Mel Gibson, Martin Scorsese y Luc Besson se si...",mel gibson martin scorsese y luc besson se sie...
5579,Según reportó el Centro Regulador de Urgencias...,segun reporto el centro regulador de urgencias...
3263,Knol pretende cubrir múltiples aspectos del co...,knol pretende cubrir multiples aspectos del co...
11895,Hay un verdadero enigma: no entendemos realmen...,hay un verdadero enigma no entendemos realment...
11031,La delegación de Colombia llegó a 28 medallas ...,la delegacion de colombia llego a medallas de ...
7687,"En una época no muy lejana, recibió críticas ...",en una epoca no muy lejana recibio criticas y ...
2092,La retención en la fuente del impuesto de rent...,la retencion en la fuente del impuesto de rent...
12185,"Las del 2018, incluidas las parlamentarias de ...",las del incluidas las parlamentarias de marzo ...
4289,1. La eliminación de la Selección del Mundial ...,la eliminacion de la seleccion del mundial de ...
3581,La construcción de las urbanizaciones de inter...,la construccion de las urbanizaciones de inter...


Paso 2: Tokenización

In [7]:
import nltk
from nltk.tokenize import word_tokenize

# Download the necessary resource
nltk.download('punkt_tab')

# Tokenización del texto preprocesado
data['tokens'] = data['contenido_preprocesado'].apply(word_tokenize)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [8]:
# Mostrar una vista previa de los tokens
data[['contenido_preprocesado', 'tokens']].sample(5)

,contenido_preprocesado,tokens
2150,es el cancer de las naciones unidas dijo el se...,"[es, el, cancer, de, las, naciones, unidas, di..."
725,justin tiberlake y jessica biel contrajeron nu...,"[justin, tiberlake, y, jessica, biel, contraje..."
11884,crece la polemica entre el gobierno y las empr...,"[crece, la, polemica, entre, el, gobierno, y, ..."
344,el de marzo se lanzara el portafolio de becas ...,"[el, de, marzo, se, lanzara, el, portafolio, d..."
11161,atletico nacional recibira a once caldas este ...,"[atletico, nacional, recibira, a, once, caldas..."


Paso 3: Eliminación de Stop Words

In [9]:
import nltk
from nltk.corpus import stopwords

# Descargar recursos necesarios de NLTK
nltk.download('stopwords')

# Definir las stop words en español
stop_words = set(stopwords.words('spanish'))

# Eliminar stop words de los tokens en el dataset limpio
data['tokens_sin_stopwords'] = data['tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)

# Mostrar una vista previa de los tokens sin stop words
data[['tokens', 'tokens_sin_stopwords']].sample(5)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,tokens,tokens_sin_stopwords
7665,"[segun, la, asociacion, de, institutores, huil...","[segun, asociacion, institutores, huilenses, a..."
1298,"[la, fiscalia, capturo, al, presunto, cabecill...","[fiscalia, capturo, presunto, cabecilla, banda..."
14024,"[tal, y, como, estaba, previsto, el, abogado, ...","[tal, previsto, abogado, juan, sebastian, rozo..."
8111,"[antes, de, aterrizar, en, lisboa, primera, et...","[aterrizar, lisboa, primera, etapa, viaje, cua..."
13694,"[el, pasado, viernes, en, la, residencia, del,...","[pasado, viernes, residencia, embajador, franc..."


Paso 4: Cálculo de TF-IDF

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Unir los tokens en una sola cadena de texto para cada documento
data['texto_sin_stopwords'] = data['tokens_sin_stopwords'].apply(lambda tokens: ' '.join(tokens))

# Calcular TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(data['texto_sin_stopwords'])

# Convertir la matriz TF-IDF a un DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# Mostrar una vista previa de la matriz TF-IDF
print(tfidf_df.head())

    aa  aaa  aaacpt  aaah  aaas  aac  aacsb  aacta  aademas  aage  ...  𝑓𝑖𝑗𝑜𝑠  \
0  0.0  0.0     0.0   0.0   0.0  0.0    0.0    0.0      0.0   0.0  ...    0.0   
1  0.0  0.0     0.0   0.0   0.0  0.0    0.0    0.0      0.0   0.0  ...    0.0   
2  0.0  0.0     0.0   0.0   0.0  0.0    0.0    0.0      0.0   0.0  ...    0.0   
3  0.0  0.0     0.0   0.0   0.0  0.0    0.0    0.0      0.0   0.0  ...    0.0   
4  0.0  0.0     0.0   0.0   0.0  0.0    0.0    0.0      0.0   0.0  ...    0.0   

    𝑙𝑎  𝑚𝑎𝑟𝑐𝑜  𝑚𝑖𝑒𝑚𝑏𝑟𝑜𝑠  𝑚𝑢𝑒𝑠𝑡𝑟𝑎  𝑝𝑐𝑒𝑙𝑢𝑙𝑎𝑟  𝑝𝑒𝑟𝑠𝑜𝑛𝑎  𝑝𝑒𝑟𝑠𝑜𝑛𝑎𝑠  𝑝𝑓𝑖𝑗𝑜   𝑝𝑖  
0  0.0    0.0       0.0      0.0       0.0      0.0       0.0    0.0  0.0  
1  0.0    0.0       0.0      0.0       0.0      0.0       0.0    0.0  0.0  
2  0.0    0.0       0.0      0.0       0.0      0.0       0.0    0.0  0.0  
3  0.0    0.0       0.0      0.0       0.0      0.0       0.0    0.0  0.0  
4  0.0    0.0       0.0      0.0       0.0      0.0       0.0    0.0  0.0  

[5 rows x 109117 columns]


Paso 5: Generación de Embeddings de Palabras con Word2Vec

In [12]:
from gensim.models import Word2Vec

# Entrenar el modelo Word2Vec
word2vec_model = Word2Vec(
    sentences=data['tokens_sin_stopwords'],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)
